# Inertial parameters

Step 5.3. Mass, center of gravity and yaw inertia, measured on the car as it
drives: slicks, LiPo, powerbank, Pi, LIDAR. Kitchen scale, 1 g resolution up to
1 kg, 5 g above.

In [1]:
import numpy as np

L = 0.173          # wheelbase [m]
R_WHEEL = 0.0299   # slick rolling radius [m]
G = 9.81

## Mass and axle loads

Whole car on the scale, then one axle at a time with the other on books at the
height of the scale, car level.

In [2]:
m = 2.015                                          # [kg], three identical readings
front = np.array([876, 863, 868, 879, 873, 875]) / 1000
rear = np.array([1155, 1155, 1160, 1155, 1155, 1160]) / 1000

W = front.mean() + rear.mean()
lf = L * rear.mean() / W
lr = L * front.mean() / W
print(f"axles {front.mean()*1000:.0f} + {rear.mean()*1000:.0f} = {W*1000:.0f} g "
      f"(whole car {m*1000:.0f} g), rear {100 * rear.mean() / W:.0f} %")
print(f"lf = {lf*1000:.1f} mm, lr = {lr*1000:.1f} mm")

axles 872 + 1157 = 2029 g (whole car 2015 g), rear 57 %
lf = 98.6 mm, lr = 74.4 mm


The axle sum is 0.7% over the whole-car reading, within the scale. Only the
ratio enters lf and lr, and the spread of the six readings moves them by 0.1 mm.

## Center of gravity height

Rear axle on the scale, front axle raised by $H$. The car pitches by
$\theta = \arcsin(H/L)$. The wheels are round, so each support force still acts
straight below its axle, and moments about the front axle give

$$W_r L\cos\theta = W\left(l_f\cos\theta + (h - r)\sin\theta\right)
\quad\Rightarrow\quad h = r + \frac{L\,\Delta W_r}{W\tan\theta}$$

A first try at H = 63 mm put the chassis, not the wheels, on the scale.
22 mm is the most the rear overhang allows.

In [3]:
H = 0.022
rear_tilted = np.array([1225, 1205, 1215, 1215, 1210, 1205]) / 1000

theta = np.arcsin(H / L)
dW = rear_tilted.mean() - rear.mean()
h = R_WHEEL + L * dW / (W * np.tan(theta))

sd_dW = np.hypot(rear_tilted.std(ddof=1), rear.std(ddof=1)) / np.sqrt(6)
sd_h = np.hypot(L * sd_dW / (W * np.tan(theta)), (h - R_WHEEL) * 0.001 / H)
print(f"theta {np.degrees(theta):.1f} deg, dW {dW*1000:.0f} g, h = {h*1000:.0f} +- {sd_h*1000:.0f} mm")

theta 7.3 deg, dW 56 g, h = 67 +- 3 mm


67 mm against the 55 mm guessed in `envelope.ipynb`. Rollover moves from 1.6 g
to 1.3 g, still far above anything the slicks can do.

## Yaw inertia, bifilar pendulum

Car hung upside down from two parallel strings tied on the center line of the
bottom plate, one ahead of the CG and one behind; flipping the car does not
change $I_z$. Strings at $d_1$, $d_2$ from the vertical through the CG carry
$mg\,d_2/(d_1+d_2)$ and $mg\,d_1/(d_1+d_2)$. A small twist $\psi$ leans each
string by $d_i\psi/L_s$, and the horizontal components give a restoring torque
$mg\,d_1d_2\,\psi/L_s$:

$$I_z\ddot\psi = -\frac{mg\,d_1d_2}{L_s}\,\psi
\quad\Rightarrow\quad I_z = \frac{mg\,d_1d_2\,T^2}{4\pi^2L_s}$$

Equal distances give the usual $d^2$. Period from phone video, 10 oscillations
per run, twist ~10 deg.

![bifilar pendulum](../media/bifilar_pendulum.jpg)

Left: strings from a bar 2.3 m up. Right: car upside down, strings in the slots
at the two ends of the bottom plate.

In [4]:
x_front, x_rear = 0.215, -0.046          # strings from the rear axle [m]
L_s = (2.310 + 2.305) / 2                  # string length [m]
d1, d2 = x_front - lr, lr - x_rear

release = np.array([1.80, 8.30, 10.60, 12.66, 22.30])
tenth = np.array([20.16, 26.82, 28.96, 31.18, 40.83])
T = (tenth - release) / 10

Iz = m * G * d1 * d2 * T.mean()**2 / (4 * np.pi**2 * L_s)
rel = np.sqrt((0.005 / m)**2 + (0.002 / d1)**2 + (0.002 / d2)**2
              + (2 * T.std(ddof=1) / np.sqrt(len(T)) / T.mean())**2 + (0.005 / L_s)**2)
print(f"T = {T.mean():.3f} s (sd {T.std(ddof=1):.3f}), d1 {d1*1000:.0f} mm, d2 {d2*1000:.0f} mm")
print(f"Iz = {Iz:.4f} +- {Iz * rel:.4f} kg m^2, radius of gyration {1000 * np.sqrt(Iz / m):.0f} mm")
print(f"5 deg twist: T = {(31.33 - 12.74) / 10:.3f} s")

T = 1.846 s (sd 0.009), d1 141 mm, d2 120 mm
Iz = 0.0125 +- 0.0003 kg m^2, radius of gyration 79 mm
5 deg twist: T = 1.859 s


The 5 deg run comes out 0.7% longer, close to the 0.5% spread between runs: no
visible dependence on amplitude, so the small-angle formula holds. The error is mostly the string positions, measured to ~2 mm.
Radius of gyration 79 mm is 0.46 of the wheelbase, in the usual 0.4-0.5 for
cars. A uniform box of the same size would give 0.018: the heavy parts sit near
the middle.

## Summary

| parameter | value | how |
|---|---|---|
| m | 2.015 kg +- 5 g | scale |
| lf | 98.6 mm +- 0.1 | axle loads |
| lr | 74.4 mm +- 0.1 | axle loads |
| h | 67 mm +- 3 | tilted axle load |
| Iz | 0.0125 kg m^2 +- 2.3% | bifilar pendulum |